In [0]:
%run ../99_utils/utils_config

UTILS CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: af765285-a358-4482-ae02-868fd033809a


In [0]:
%run ../99_utils/utils_quality

UTILS CONFIG LOADED SUCCESSFULLY
PROJECT_NAME: brazil_legislative_analytics
PROJECT_VERSION: v1.0.0
PROJECT_ENVIRONMENT: dev
CATALOG_NAME: brazil_legislative_analytics
RUN_ID: 5f33f785-5a92-460f-8a40-7e42a44b8bdf


utils_quality loaded successfully.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02 Quality — Silver Checks
# MAGIC
# MAGIC Executes data quality validations for Silver tables in the Brazil Legislative Analytics Medallion project.
# MAGIC
# MAGIC ## Purpose
# MAGIC This notebook validates whether Silver curated outputs are available, structurally consistent and ready for Gold dimensional modeling.
# MAGIC
# MAGIC ## Scope
# MAGIC Silver quality checks focus on:
# MAGIC - table existence
# MAGIC - required traceability columns
# MAGIC - minimum record availability
# MAGIC - duplicated hash records
# MAGIC - business key availability
# MAGIC - controlled exception handling per entity
# MAGIC
# MAGIC ## Persistence
# MAGIC Validation results are persisted into:
# MAGIC
# MAGIC ```text
# MAGIC audit.aud_log_qualidade_dados
# MAGIC ```
# MAGIC
# MAGIC ## Execution Policy
# MAGIC During early development, Silver tables may not exist yet.
# MAGIC In this case, validations are persisted as evidence, but the notebook does not block execution.
# MAGIC
# MAGIC Set `FAIL_ON_ERROR = True` when Silver processing is active and quality checks must block the pipeline.
# MAGIC
# MAGIC ## Documentation Standard
# MAGIC - Python functions and variables are written in English.
# MAGIC - Table and field names follow Portuguese mnemonic standards.
# MAGIC - Comments and documentation are written in English.

# COMMAND ----------

# MAGIC %run ../99_utils/utils_config

# COMMAND ----------

# MAGIC %run ../99_utils/utils_quality

# COMMAND ----------

from datetime import datetime
from pyspark.sql import DataFrame
from pyspark.sql import functions as F

# COMMAND ----------

print("=" * 90)
print("BRAZIL LEGISLATIVE ANALYTICS MEDALLION")
print("02 - QUALITY SILVER CHECKS")
print("=" * 90)
print(f"Execution Timestamp: {datetime.now()}")
print(f"Catalog: {CATALOG_NAME}")
print(f"Layer: {SCHEMA_SILVER}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# QUALITY CONFIGURATION
# ============================================================

NOTEBOOK_NAME = "02_quality_silver_checks"
LAYER_NAME = "silver"

# During early development, Silver tables may not exist yet.
# Set to True when Silver processing is active and quality checks
# must block the pipeline.
FAIL_ON_ERROR = False

DATA_QUALITY_LOG_TABLE = (
    f"{CATALOG_NAME}."
    f"{SCHEMA_AUDIT}."
    f"{AUD_TB_LOG_QUALIDADE_DADOS}"
)

SILVER_ENTITY_TABLES = SILVER_TABLES

SILVER_REQUIRED_COLUMNS = [
    "aud_id_execucao",
    "aud_dh_processamento",
    "aud_tx_versao_pipeline",
    "aud_tx_hash_registro",
]

SILVER_BUSINESS_KEYS = {
    "deputados": ["id_deputado"],
    "partidos": ["id_partido"],
    "estados": ["sigla_uf"],
    "frentes": ["id_frente"],
    "frentes_membros": ["id_frente", "id_deputado"],
    "eventos": ["id_evento"],
    "votacoes": ["id_votacao"],
    "votos": ["id_votacao", "id_deputado"],
    "despesas_ceap": ["id_deputado", "id_documento"],
    "fornecedores": ["cnpj_cpf_fornecedor"],
    "cpis": ["id_cpi"],
    "cpi_eventos": ["id_cpi", "id_evento"],
    "proposicoes": ["id_proposicao"],
}

quality_results = []

# COMMAND ----------

# ============================================================
# QUALITY HELPERS
# ============================================================

def add_quality_result(
    rule_name: str,
    rule_description: str,
    validation_status: str,
    total_records: int,
    invalid_records: int,
    invalid_percentage: float,
    message: str,
    entity_name: str,
    target_table: str,
) -> None:
    """
    Adds a quality validation result to the in-memory result list.
    """

    quality_results.append({
        "nome_regra": rule_name,
        "descricao_regra": rule_description,
        "status_validacao": validation_status,
        "total_registros": int(total_records) if total_records is not None else 0,
        "registros_invalidos": int(invalid_records) if invalid_records is not None else 0,
        "percentual_invalidos": float(invalid_percentage) if invalid_percentage is not None else 0.0,
        "mensagem": message,
        "entity_name": entity_name,
        "target_table": target_table,
    })

# COMMAND ----------

def add_exception_result(
    entity_name: str,
    target_table: str,
    error: Exception,
) -> None:
    """
    Adds a controlled exception result to the quality result list.
    """

    add_quality_result(
        rule_name="silver_quality_exception",
        rule_description="Captures unexpected errors during Silver quality validation.",
        validation_status=QUALITY_FAILED,
        total_records=1,
        invalid_records=1,
        invalid_percentage=100.0,
        message=f"Unexpected error during Silver quality validation: {str(error)}",
        entity_name=entity_name,
        target_table=target_table,
    )

# COMMAND ----------

def table_exists(
    full_table_name: str,
) -> bool:
    """
    Checks whether a fully qualified table exists.
    """

    try:
        return spark.catalog.tableExists(full_table_name)

    except Exception:
        return False

# COMMAND ----------

def get_table_dataframe(
    full_table_name: str,
) -> DataFrame:
    """
    Reads a table into a Spark DataFrame.
    """

    return spark.table(full_table_name)

# COMMAND ----------

def count_records(
    dataframe: DataFrame,
) -> int:
    """
    Counts records from a Spark DataFrame.
    """

    return dataframe.count()

# COMMAND ----------

def validate_table_exists(
    entity_name: str,
    full_table_name: str,
) -> bool:
    """
    Validates whether a Silver table exists.
    """

    exists = table_exists(full_table_name)

    add_quality_result(
        rule_name="silver_table_exists",
        rule_description="Validates whether the Silver table exists.",
        validation_status=QUALITY_PASSED if exists else QUALITY_FAILED,
        total_records=1,
        invalid_records=0 if exists else 1,
        invalid_percentage=0.0 if exists else 100.0,
        message=(
            "Silver table exists."
            if exists
            else "Silver table does not exist."
        ),
        entity_name=entity_name,
        target_table=full_table_name,
    )

    return exists

# COMMAND ----------

def validate_minimum_records(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> int:
    """
    Validates whether a Silver table contains at least one record.
    """

    total_records = count_records(dataframe)
    invalid_records = 0 if total_records > 0 else 1

    add_quality_result(
        rule_name="silver_minimum_records",
        rule_description="Validates whether the Silver table contains at least one record.",
        validation_status=QUALITY_PASSED if total_records > 0 else QUALITY_WARNING,
        total_records=total_records,
        invalid_records=invalid_records,
        invalid_percentage=0.0 if total_records > 0 else 100.0,
        message=f"Silver table record count: {total_records}",
        entity_name=entity_name,
        target_table=full_table_name,
    )

    return total_records

# COMMAND ----------

def validate_traceability_columns(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates required Silver traceability columns.
    """

    result = validate_required_columns(
        dataframe=dataframe,
        required_columns=SILVER_REQUIRED_COLUMNS,
    )

    add_quality_result(
        rule_name="silver_required_traceability_columns",
        rule_description="Validates required Silver traceability columns.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def validate_business_key_columns(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates whether expected business key columns exist.
    """

    key_columns = SILVER_BUSINESS_KEYS.get(entity_name, [])

    if not key_columns:
        add_quality_result(
            rule_name="silver_business_key_mapping",
            rule_description="Validates whether the Silver entity has a configured business key.",
            validation_status=QUALITY_WARNING,
            total_records=1,
            invalid_records=0,
            invalid_percentage=0.0,
            message=f"No business key mapping configured for entity: {entity_name}",
            entity_name=entity_name,
            target_table=full_table_name,
        )
        return

    result = validate_required_columns(
        dataframe=dataframe,
        required_columns=key_columns,
    )

    add_quality_result(
        rule_name="silver_business_key_columns",
        rule_description="Validates expected Silver business key columns.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def validate_business_key_nulls(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates null values in business key columns.
    """

    key_columns = [
        column
        for column in SILVER_BUSINESS_KEYS.get(entity_name, [])
        if column in dataframe.columns
    ]

    if not key_columns:
        return

    results = validate_nulls(
        dataframe=dataframe,
        columns=key_columns,
    )

    for result in results:
        add_quality_result(
            rule_name=f"silver_{result['nome_regra']}",
            rule_description=result["descricao_regra"],
            validation_status=result["status_validacao"],
            total_records=result["total_registros"],
            invalid_records=result["registros_invalidos"],
            invalid_percentage=result["percentual_invalidos"],
            message=result["mensagem"],
            entity_name=entity_name,
            target_table=full_table_name,
        )

# COMMAND ----------

def validate_hash_duplicates(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates duplicated records based on the Silver record hash.
    """

    if "aud_tx_hash_registro" not in dataframe.columns:

        add_quality_result(
            rule_name="silver_hash_duplicate_check",
            rule_description="Validates duplicated records based on the Silver record hash.",
            validation_status=QUALITY_FAILED,
            total_records=1,
            invalid_records=1,
            invalid_percentage=100.0,
            message="Column aud_tx_hash_registro does not exist.",
            entity_name=entity_name,
            target_table=full_table_name,
        )

        return

    result = validate_duplicates(
        dataframe=dataframe,
        key_columns=["aud_tx_hash_registro"],
    )

    add_quality_result(
        rule_name="silver_hash_duplicate_check",
        rule_description="Validates duplicated records based on the Silver record hash.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def validate_business_key_duplicates(
    dataframe: DataFrame,
    entity_name: str,
    full_table_name: str,
) -> None:
    """
    Validates duplicated records based on configured business key columns.
    """

    key_columns = [
        column
        for column in SILVER_BUSINESS_KEYS.get(entity_name, [])
        if column in dataframe.columns
    ]

    if not key_columns:
        return

    result = validate_duplicates(
        dataframe=dataframe,
        key_columns=key_columns,
    )

    add_quality_result(
        rule_name="silver_business_key_duplicate_check",
        rule_description="Validates duplicated records based on configured business key columns.",
        validation_status=result["status_validacao"],
        total_records=result["total_registros"],
        invalid_records=result["registros_invalidos"],
        invalid_percentage=result["percentual_invalidos"],
        message=result["mensagem"],
        entity_name=entity_name,
        target_table=full_table_name,
    )

# COMMAND ----------

def run_entity_checks(
    entity_name: str,
    table_name: str,
) -> None:
    """
    Executes all Silver quality checks for a single entity.
    """

    full_table_name = get_silver_table(table_name)

    print("=" * 90)
    print(f"Running Silver quality checks for: {full_table_name}")
    print("=" * 90)

    try:

        if not validate_table_exists(
            entity_name=entity_name,
            full_table_name=full_table_name,
        ):
            return

        dataframe = get_table_dataframe(full_table_name)

        validate_minimum_records(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_traceability_columns(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_business_key_columns(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_business_key_nulls(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_hash_duplicates(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

        validate_business_key_duplicates(
            dataframe=dataframe,
            entity_name=entity_name,
            full_table_name=full_table_name,
        )

    except Exception as error:

        add_exception_result(
            entity_name=entity_name,
            target_table=full_table_name,
            error=error,
        )

# COMMAND ----------

def build_silver_quality_log() -> DataFrame:
    """
    Builds the final Silver quality log DataFrame.
    """

    if not quality_results:

        add_quality_result(
            rule_name="silver_quality_no_results",
            rule_description="Validates whether Silver quality checks produced results.",
            validation_status=QUALITY_WARNING,
            total_records=0,
            invalid_records=0,
            invalid_percentage=0.0,
            message="No Silver quality results were generated.",
            entity_name="silver",
            target_table=DATA_QUALITY_LOG_TABLE,
        )

    quality_base_df = spark.createDataFrame(
        quality_results
    )

    return (
        quality_base_df
        .withColumn("qlt_id_log", F.expr("uuid()"))
        .withColumn("aud_id_execucao", F.lit(RUN_ID))
        .withColumn("aud_tx_nome_projeto", F.lit(PROJECT_NAME))
        .withColumn("aud_tx_versao_pipeline", F.lit(PROJECT_VERSION))
        .withColumn("aud_tx_ambiente", F.lit(PROJECT_ENVIRONMENT))
        .withColumn("aud_tx_nome_notebook", F.lit(NOTEBOOK_NAME))
        .withColumn("aud_tx_nome_camada", F.lit(LAYER_NAME))
        .withColumn("aud_tx_nome_entidade", F.col("entity_name"))
        .withColumn("aud_tx_tabela_destino", F.col("target_table"))
        .withColumn("qlt_tx_nome_regra", F.col("nome_regra"))
        .withColumn("qlt_tx_descricao_regra", F.col("descricao_regra"))
        .withColumn("qlt_tx_status_validacao", F.col("status_validacao"))
        .withColumn("qlt_qt_total_registros", F.col("total_registros"))
        .withColumn("qlt_qt_registros_invalidos", F.col("registros_invalidos"))
        .withColumn("qlt_pc_registros_invalidos", F.col("percentual_invalidos"))
        .withColumn("qlt_dh_validacao", F.current_timestamp())
        .withColumn("qlt_tx_mensagem", F.col("mensagem"))
        .select(
            "qlt_id_log",
            "aud_id_execucao",
            "aud_tx_nome_projeto",
            "aud_tx_versao_pipeline",
            "aud_tx_ambiente",
            "aud_tx_nome_notebook",
            "aud_tx_nome_camada",
            "aud_tx_nome_entidade",
            "aud_tx_tabela_destino",
            "qlt_tx_nome_regra",
            "qlt_tx_descricao_regra",
            "qlt_tx_status_validacao",
            "qlt_qt_total_registros",
            "qlt_qt_registros_invalidos",
            "qlt_pc_registros_invalidos",
            "qlt_dh_validacao",
            "qlt_tx_mensagem",
        )
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Execute Silver Quality Checks

# COMMAND ----------

for entity_name, table_name in SILVER_ENTITY_TABLES.items():

    run_entity_checks(
        entity_name=entity_name,
        table_name=table_name,
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Persist Quality Results

# COMMAND ----------

quality_log_df = build_silver_quality_log()

quality_log_df.write.mode(
    "append"
).saveAsTable(DATA_QUALITY_LOG_TABLE)

print(
    f"Silver quality results persisted into: "
    f"{DATA_QUALITY_LOG_TABLE}"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Display Quality Results

# COMMAND ----------

display(quality_log_df)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Quality Summary

# COMMAND ----------

failed_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'FAILED'")
    .count()
)

warning_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'WARNING'")
    .count()
)

passed_count = (
    quality_log_df
    .filter("qlt_tx_status_validacao = 'PASSED'")
    .count()
)

print("=" * 90)
print("SILVER QUALITY SUMMARY")
print("=" * 90)
print(f"Passed validations: {passed_count}")
print(f"Warning validations: {warning_count}")
print(f"Failed validations: {failed_count}")
print("=" * 90)

# COMMAND ----------

# ============================================================
# QUALITY EXECUTION POLICY
# ============================================================

if failed_count > 0 and FAIL_ON_ERROR:

    raise Exception(
        f"Silver quality validation failed with "
        f"{failed_count} failed validation(s)."
    )

if failed_count > 0:

    print(
        f"WARNING: Silver quality validation finished with "
        f"{failed_count} failed validation(s). "
        "This is expected if Silver tables have not been created yet."
    )

print("SILVER QUALITY CHECKS COMPLETED")

BRAZIL LEGISLATIVE ANALYTICS MEDALLION
02 - QUALITY SILVER CHECKS
Execution Timestamp: 2026-05-20 03:13:39.704105
Catalog: brazil_legislative_analytics
Layer: silver
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_deputados
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_partidos
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_estados
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_frentes
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_frentes_membros
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_eventos
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_votacoes
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_votos
Running Silver quality checks for: brazil_legislative_analytics.silver.slv_despesas_ceap
Running Silver quality checks for: brazil_legislative_analytics.silver.

qlt_id_log,aud_id_execucao,aud_tx_nome_projeto,aud_tx_versao_pipeline,aud_tx_ambiente,aud_tx_nome_notebook,aud_tx_nome_camada,aud_tx_nome_entidade,aud_tx_tabela_destino,qlt_tx_nome_regra,qlt_tx_descricao_regra,qlt_tx_status_validacao,qlt_qt_total_registros,qlt_qt_registros_invalidos,qlt_pc_registros_invalidos,qlt_dh_validacao,qlt_tx_mensagem
a6330904-f258-4890-b1a2-019fef7986d2,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,deputados,brazil_legislative_analytics.silver.slv_deputados,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
89ff9a8a-a699-4c97-acf6-4f5524231e4a,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,partidos,brazil_legislative_analytics.silver.slv_partidos,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
736fc51e-6aae-42d9-bc8e-762ea0afd530,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,estados,brazil_legislative_analytics.silver.slv_estados,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
3301d911-2704-4876-be91-36b296e3f0ce,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,frentes,brazil_legislative_analytics.silver.slv_frentes,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
2e78da1b-99a2-4631-a78e-19b60c7a8331,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,frentes_membros,brazil_legislative_analytics.silver.slv_frentes_membros,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
d7d0f031-6c74-4d99-8294-6eefae784ee2,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,eventos,brazil_legislative_analytics.silver.slv_eventos,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
8b1eec80-ed06-4ce7-abb9-52b8133b90eb,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,votacoes,brazil_legislative_analytics.silver.slv_votacoes,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
83580961-fc93-4e6b-8b12-3a0d55bad822,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,votos,brazil_legislative_analytics.silver.slv_votos,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
a7bf7481-ce3d-4b16-8791-27f6250e64f5,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,despesas_ceap,brazil_legislative_analytics.silver.slv_despesas_ceap,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.
828f37cf-de03-4c97-984f-db2f5fb8fe43,5f33f785-5a92-460f-8a40-7e42a44b8bdf,brazil_legislative_analytics,v1.0.0,dev,02_quality_silver_checks,silver,fornecedores,brazil_legislative_analytics.silver.slv_fornecedores,silver_table_exists,Validates whether the Silver table exists.,FAILED,1,1,100.0,2026-05-20T03:13:44.248Z,Silver table does not exist.


SILVER QUALITY SUMMARY
Passed validations: 0
Warning validations: 0
Failed validations: 13
SILVER QUALITY CHECKS COMPLETED
